In [ ]:
import tqdm, tifffile
import numpy as np
import os, sys
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc

# Include src directory in the path dynamically
notebook_dir = os.path.abspath("")
src_path = os.path.join(notebook_dir, "../src")
sys.path.append(src_path)

from constants import MODALITY_ALIGNMENT, MODALITY_PREPROCESSING


In [ ]:
PATH = "/staging/leuven/stg_00077/projects/Lorenzo/FOCUS/p_PDA/d_C1"
REFERENCE_MODALITY_NAME = 'Raman'
TARGET_MODALITY_NAME = 'MSI'
SAMPLE_ID = "PDAC010N0"

In [ ]:
# Load the OME TIFF image from the reference
reference_image_path = MODALITY_PREPROCESSING(PATH, SAMPLE_ID, REFERENCE_MODALITY_NAME, "ome.tiff")
with tifffile.TiffFile(reference_image_path) as tif:
    reference_image = tif.asarray()

reference_image = reference_image.transpose(1, 2, 0)  # Convert from CHW to HWC format
reference_image = np.mean(reference_image, axis=2)  # Convert to grayscale by averaging channels

print(f"Reference image shape: {reference_image.shape}")

In [ ]:
# Load the target modality AnnData
target_anndata_path = MODALITY_ALIGNMENT(PATH, SAMPLE_ID, TARGET_MODALITY_NAME, "h5ad")
target_adata = ad.read_h5ad(target_anndata_path)
print(f"Target AnnData shape: {target_adata.shape}")

In [ ]:
# Plot the reference image with overlaid target modality spatial points
plt.figure(figsize=(20, 20))
plt.imshow(reference_image, cmap='gray')
target_coords = target_adata.obsm[f'{REFERENCE_MODALITY_NAME}_spatial']  # Assuming spatial coordinates are stored here
plt.scatter(target_coords[:, 0], target_coords[:, 1], c='red', s=3, alpha=0.5)
plt.title(f"Aligned {TARGET_MODALITY_NAME} data on {REFERENCE_MODALITY_NAME} image for {SAMPLE_ID}")
plt.show()